In [14]:
import pandas as pd
import statsmodels.api as sm
from statsmodels.iolib.summary2 import summary_col
from tabulate import tabulate
import numpy as np

In [15]:
train_panel_in = "../../../data/final/train_panel.parquet"
bike_panel_in = "../../../data/final/bike_panel.parquet"
uber_panel_in = "../../../data/final/uber_panel.parquet"

did_out = "../../../reports/replication/did.md"
fe_out = "../../../reports/replication/fe.md"

train_panel_out = "../../../data/final/train_model.parquet"
bike_panel_out = "../../../data/final/bike_model.parquet"
uber_panel_out = "../../../data/final/uber_model.parquet"


In [16]:
train_panel = pd.read_parquet(train_panel_in)
bike_panel = pd.read_parquet(bike_panel_in)
uber_panel = pd.read_parquet(uber_panel_in)

# Preprocessing

In [17]:
dotw_names = {f"C(dotw)[T.{i}]":j for i,j in zip([1,2,3,4],['Tuesday','Wednesday','Thursday','Friday'])}

In [18]:
# The DNC was only 4 days, not even friday. so weekends are uninformative.
# TODO: don't hard-code because we want to include more days for robustness check
train_panel = train_panel[(0 <= train_panel['dotw']) & (train_panel['dotw'] <= 3)]
bike_panel = bike_panel[(0 <= bike_panel['dotw']) & (bike_panel['dotw'] <= 3)]
uber_panel = uber_panel[(0 <= uber_panel['dotw']) & (uber_panel['dotw'] <= 3)]

In [19]:
train_panel.to_parquet(train_panel_out)
bike_panel.to_parquet(bike_panel_out)
uber_panel.to_parquet(uber_panel_out)

# DiffNDiff

In [20]:
covariates = ['distance_loop', 'distance_ohare', 'distance_midway',
        'train_distance', 'train_contained', 'bike_distance', 'bike_contained', 'bus_distance', 'bus_contained',
        'landarea', 'TOTAL_POPULATION', 'MEDIAN_HOUSEHOLD_INCOME', 'C(dotw)', 'time']
train_covs = list(set(covariates) - set(['train_contained']))
bike_covs = list(set(covariates) - set(['bike_contained']))

In [21]:
def mask_colinear(df, groupby, coef):
    test = df.groupby(groupby)[coef].nunique() > 1
    mask = df[groupby].isin(test[test].index)
    if not all(test):
        print(f"Dropping {sum(~test)} {groupby}, {sum(~mask)} obs invariant to {coef}")
    return mask

def did_colinearity(df):
    mask = np.full_like(df.index, True)
    mask &= mask_colinear(df, 'id', 'DNC')
    mask &= mask_colinear(df, 'date', 'UCMP')
    df = df.loc[mask]
    return df

def model_transit_did(df, covs):
    df = df.sort_values('date')
    unit_obs = df.groupby('id').size()
    df['weight'] = df['id'].map(unit_obs).apply(lambda x: 1/x)
    cov_term = " + ".join(covs)
    # Assert that spatial units have time variance (only need to test group inclusion, not every variable.)
    df = did_colinearity(df)

    formula = f"""logrides ~ UCMP * DNC + {cov_term}"""
    did_model = sm.WLS.from_formula(formula, df, weights=df['weight']).fit(cov_type='cluster',cov_kwds={'groups': df['id']})
    return did_model

In [22]:
bike_did_model = model_transit_did(bike_panel, bike_covs)
bike_did_model.summary()

Dropping 458 id, 2840 obs invariant to DNC


/Users/eric/Repos/DNCTransit/.venv/lib/python3.10/site-packages/numpy/linalg/_linalg.py:3383: RuntimeWarning: divide by zero encountered in matmul
  return _core_matmul(x1, x2)
/Users/eric/Repos/DNCTransit/.venv/lib/python3.10/site-packages/numpy/linalg/_linalg.py:3383: RuntimeWarning: overflow encountered in matmul
  return _core_matmul(x1, x2)
/Users/eric/Repos/DNCTransit/.venv/lib/python3.10/site-packages/numpy/linalg/_linalg.py:3383: RuntimeWarning: invalid value encountered in matmul
  return _core_matmul(x1, x2)


<class 'statsmodels.iolib.summary.Summary'>
"""
                            WLS Regression Results                            
==============================================================================
Dep. Variable:               logrides   R-squared:                       0.688
Model:                            WLS   Adj. R-squared:                  0.688
Method:                 Least Squares   F-statistic:                     165.4
Date:                Sun, 11 Oct 2026   Prob (F-statistic):          5.26e-292
Time:                        17:47:55   Log-Likelihood:                -58248.
No. Observations:               43303   AIC:                         1.165e+05
Df Residuals:                   43284   BIC:                         1.167e+05
Df Model:                          18                                         
Covariance Type:              cluster                                         
===========================================================================================
                              coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------
Intercept                   1.0736      0.165      6.520      0.000       0.751       1.396
C(dotw)[T.1]               -0.0034      0.013     -0.258      0.797      -0.029       0.022
C(dotw)[T.2]                0.0802      0.015      5.274      0.000       0.050       0.110
C(dotw)[T.3]                0.0498      0.014      3.471      0.001       0.022       0.078
UCMP                       -0.0359      0.111     -0.324      0.746      -0.253       0.181
DNC                        -0.0494      0.024     -2.041      0.041      -0.097      -0.002
UCMP:DNC                    0.3212      0.055      5.798      0.000       0.213       0.430
train_contained            -0.1108      0.098     -1.130      0.259      -0.303       0.081
MEDIAN_HOUSEHOLD_INCOME  1.511e-06   8.32e-07      1.818      0.069   -1.18e-07    3.14e-06
bus_distance                0.2438      0.281      0.867      0.386      -0.308       0.795
landarea                    0.0429      0.053      0.802      0.422      -0.062       0.148
TOTAL_POPULATION         7.483e-07   1.79e-05      0.042      0.967   -3.44e-05    3.59e-05
train_distance             -0.0522      0.032     -1.618      0.106      -0.115       0.011
time                        0.0002      0.000      0.739      0.460      -0.000       0.001
distance_ohare              0.0925      0.006     16.687      0.000       0.082       0.103
bus_contained              -0.0074      0.003     -2.259      0.024      -0.014      -0.001
bike_distance               0.4825      0.205      2.356      0.018       0.081       0.884
distance_loop              -0.3764      0.009    -40.531      0.000      -0.395      -0.358
distance_midway             0.2252      0.009     23.928      0.000       0.207       0.244
==============================================================================
Omnibus:                     4523.098   Durbin-Watson:                   1.942
Prob(Omnibus):                  0.000   Jarque-Bera (JB):            14730.261
Skew:                          -0.535   Prob(JB):                         0.00
Kurtosis:                       5.650   Cond. No.                     1.33e+06
==============================================================================

Notes:
[1] Standard Errors are robust to cluster correlation (cluster)
[2] The condition number is large, 1.33e+06. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [23]:
train_did_model = model_transit_did(train_panel, train_covs)
train_did_model.summary()

/Users/eric/Repos/DNCTransit/.venv/lib/python3.10/site-packages/numpy/linalg/_linalg.py:3383: RuntimeWarning: divide by zero encountered in matmul
  return _core_matmul(x1, x2)
/Users/eric/Repos/DNCTransit/.venv/lib/python3.10/site-packages/numpy/linalg/_linalg.py:3383: RuntimeWarning: overflow encountered in matmul
  return _core_matmul(x1, x2)
/Users/eric/Repos/DNCTransit/.venv/lib/python3.10/site-packages/numpy/linalg/_linalg.py:3383: RuntimeWarning: invalid value encountered in matmul
  return _core_matmul(x1, x2)


<class 'statsmodels.iolib.summary.Summary'>
"""
                            WLS Regression Results                            
==============================================================================
Dep. Variable:               logrides   R-squared:                       0.328
Model:                            WLS   Adj. R-squared:                  0.326
Method:                 Least Squares   F-statistic:                     64.91
Date:                Sun, 11 Oct 2026   Prob (F-statistic):           5.91e-59
Time:                        17:47:55   Log-Likelihood:                -9583.7
No. Observations:                7932   AIC:                         1.921e+04
Df Residuals:                    7913   BIC:                         1.934e+04
Df Model:                          18                                         
Covariance Type:              cluster                                         
===========================================================================================
                              coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------
Intercept                   6.7335      0.510     13.194      0.000       5.733       7.734
C(dotw)[T.1]                0.1549      0.007     21.651      0.000       0.141       0.169
C(dotw)[T.2]                0.1659      0.006     28.741      0.000       0.155       0.177
C(dotw)[T.3]                0.1398      0.005     26.027      0.000       0.129       0.150
UCMP                       -1.0972      0.592     -1.854      0.064      -2.257       0.063
DNC                        -0.1142      0.015     -7.499      0.000      -0.144      -0.084
UCMP:DNC                    0.8057      0.481      1.676      0.094      -0.137       1.748
bike_contained           8.512e-05    5.2e-05      1.637      0.102   -1.68e-05       0.000
MEDIAN_HOUSEHOLD_INCOME -3.824e-06   2.18e-06     -1.752      0.080    -8.1e-06    4.54e-07
bus_distance                0.1678      0.294      0.571      0.568      -0.408       0.744
landarea                   -0.0260      0.034     -0.758      0.448      -0.093       0.041
TOTAL_POPULATION         -2.28e-05   4.42e-05     -0.516      0.606      -0.000    6.38e-05
train_distance              1.1341      0.272      4.167      0.000       0.601       1.668
time                        0.0008      0.000      3.981      0.000       0.000       0.001
distance_ohare              0.0196      0.025      0.781      0.435      -0.030       0.069
bus_contained              -0.0004      0.000     -0.893      0.372      -0.001       0.000
bike_distance               0.0554      0.204      0.272      0.786      -0.344       0.454
distance_loop              -0.1981      0.029     -6.824      0.000      -0.255      -0.141
distance_midway             0.1013      0.023      4.314      0.000       0.055       0.147
==============================================================================
Omnibus:                     5438.762   Durbin-Watson:                   2.109
Prob(Omnibus):                  0.000   Jarque-Bera (JB):           280447.464
Skew:                          -2.686   Prob(JB):                         0.00
Kurtosis:                      31.630   Cond. No.                     1.39e+06
==============================================================================

Notes:
[1] Standard Errors are robust to cluster correlation (cluster)
[2] The condition number is large, 1.39e+06. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [24]:
uber_did_model = model_transit_did(uber_panel, covariates)
uber_did_model.summary()

Dropping 130 id, 762 obs invariant to DNC


/Users/eric/Repos/DNCTransit/.venv/lib/python3.10/site-packages/numpy/linalg/_linalg.py:3383: RuntimeWarning: divide by zero encountered in matmul
  return _core_matmul(x1, x2)
/Users/eric/Repos/DNCTransit/.venv/lib/python3.10/site-packages/numpy/linalg/_linalg.py:3383: RuntimeWarning: overflow encountered in matmul
  return _core_matmul(x1, x2)
/Users/eric/Repos/DNCTransit/.venv/lib/python3.10/site-packages/numpy/linalg/_linalg.py:3383: RuntimeWarning: invalid value encountered in matmul
  return _core_matmul(x1, x2)


<class 'statsmodels.iolib.summary.Summary'>
"""
                            WLS Regression Results                            
==============================================================================
Dep. Variable:               logrides   R-squared:                       0.682
Model:                            WLS   Adj. R-squared:                  0.682
Method:                 Least Squares   F-statistic:                     215.0
Date:                Sun, 11 Oct 2026   Prob (F-statistic):               0.00
Time:                        17:47:55   Log-Likelihood:                -78263.
No. Observations:               54949   AIC:                         1.566e+05
Df Residuals:                   54929   BIC:                         1.567e+05
Df Model:                          19                                         
Covariance Type:              cluster                                         
===========================================================================================
                              coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------
Intercept                   1.8831      0.164     11.460      0.000       1.561       2.205
C(dotw)[T.1]                0.1311      0.009     15.167      0.000       0.114       0.148
C(dotw)[T.2]                0.2168      0.009     24.106      0.000       0.199       0.234
C(dotw)[T.3]                0.3149      0.010     30.810      0.000       0.295       0.335
UCMP                       -0.0958      0.166     -0.577      0.564      -0.421       0.229
DNC                        -0.0499      0.018     -2.706      0.007      -0.086      -0.014
UCMP:DNC                    0.2019      0.083      2.445      0.014       0.040       0.364
distance_loop              -0.2930      0.016    -18.260      0.000      -0.324      -0.262
distance_ohare              0.0872      0.007     12.235      0.000       0.073       0.101
distance_midway             0.1367      0.008     16.638      0.000       0.121       0.153
train_distance             -0.2412      0.035     -6.810      0.000      -0.311      -0.172
train_contained             0.0107      0.149      0.072      0.943      -0.281       0.303
bike_distance              -0.2868      0.089     -3.226      0.001      -0.461      -0.113
bike_contained              0.0004      0.000      3.817      0.000       0.000       0.001
bus_distance                0.5042      0.093      5.429      0.000       0.322       0.686
bus_contained               0.0181      0.004      4.664      0.000       0.011       0.026
landarea                    0.7278      0.164      4.432      0.000       0.406       1.050
TOTAL_POPULATION            0.0002   2.08e-05     11.307      0.000       0.000       0.000
MEDIAN_HOUSEHOLD_INCOME  1.057e-06   8.39e-07      1.260      0.208   -5.88e-07     2.7e-06
time                        0.0009      0.000      8.255      0.000       0.001       0.001
==============================================================================
Omnibus:                     2507.520   Durbin-Watson:                   1.424
Prob(Omnibus):                  0.000   Jarque-Bera (JB):             7236.339
Skew:                          -0.198   Prob(JB):                         0.00
Kurtosis:                       4.733   Cond. No.                     1.65e+06
==============================================================================

Notes:
[1] Standard Errors are robust to cluster correlation (cluster)
[2] The condition number is large, 1.65e+06. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [25]:
headers = ["DiD (Uber)", "DiD (Train)", "DiD (Bike)"]

top_covs = ['UCMP','DNC','UCMP:DNC']
contained_covs = sorted([c for c in covariates if 'contain' in c])
dist_covs = sorted([c for c in covariates if 'distance' in c])
other_covs = sorted(list(set(covariates) - set(top_covs) - set(contained_covs) - set(dist_covs)))
cov_order = top_covs + contained_covs + dist_covs + other_covs

did_summary = summary_col([uber_did_model, train_did_model, bike_did_model], 
            model_names=headers, 
            info_dict={"N":lambda x:(x.nobs)},
            stars=True,
            regressor_order=cov_order,
            drop_omitted=False)

did_summary.tables[0].index = (did_summary.tables[0].index
                               .str.replace('DNC','During DNC')
                               .str.replace('UCMP','Near DNC')
                               .str.replace('time','time trend')
                               .str.replace('MEDIAN_HOUSEHOLD_INCOME', 'Median Household Income')
                               .str.replace('TOTAL_POPULATION', 'Total Population')
                               .str.replace('bike_contained', 'N Bike Stations')
                               .str.replace('bus_contained', 'N Bus Stations')
                               .str.replace('train_contained', 'N Train Stations')
                               .str.replace('train_distance', 'Nearest Train Station')
                               .str.replace('bus_distance', 'Nearest Bus Stop')
                               .str.replace('bike_distance', 'Nearest Bike Station')
                               .str.replace('distance_loop', 'Distance to Loop')
                               .str.replace('distance_midway', 'Distance to Midway')
                               .str.replace('distance_ohare', 'Distance to OHare')
                               .map(lambda x: dotw_names[x] if x in dotw_names else x))

did_table = tabulate(did_summary.tables[0],
                   headers=[""] + headers,
                   showindex=True,
                   tablefmt='github')
print(did_table)
with open(did_out,"w") as f:
    f.write(did_table)

!cp {did_out} ../../../../eric-mc2-cv/static/uploads/

|                         | DiD (Uber)   | DiD (Train)   | DiD (Bike)   |
|-------------------------|--------------|---------------|--------------|
| Near DNC                | -0.0958      | -1.0972*      | -0.0359      |
|                         | (0.1659)     | (0.5918)      | (0.1106)     |
| During DNC              | -0.0499***   | -0.1142***    | -0.0494**    |
|                         | (0.0184)     | (0.0152)      | (0.0242)     |
| Near DNC:During DNC     | 0.2019**     | 0.8057*       | 0.3212***    |
|                         | (0.0825)     | (0.4807)      | (0.0554)     |
| N Bike Stations         | 0.0004***    | 0.0001        |              |
|                         | (0.0001)     | (0.0001)      |              |
| N Bus Stations          | 0.0181***    | -0.0004       | -0.0074**    |
|                         | (0.0039)     | (0.0005)      | (0.0033)     |
| N Train Stations        | 0.0107       |               | -0.1108      |
|                         | (0.1489)  

## DiD Impact

In [26]:
beta1 = did_summary.tables[0].xs('During DNC').str.replace('*','').astype(float)
main_pct = np.exp(beta1) - 1
pct1 = main_pct.apply("{:.1%}".format)

beta2 = did_summary.tables[0].xs('Near DNC').str.replace('*','').astype(float)
main_pct = np.exp(beta2) - 1
pct2 = main_pct.apply("{:.1%}".format)

beta3 = did_summary.tables[0].xs('Near DNC:During DNC').str.replace('*','').astype(float)
main_pct = np.exp(beta3) - 1
pct3 = main_pct.apply("{:.1%}".format)
pd.concat([pct1,pct2,pct3],axis=1)

,During DNC,Near DNC,Near DNC:During DNC
DiD (Uber),-4.9%,-9.1%,22.4%
DiD (Train),-10.8%,-66.6%,123.8%
DiD (Bike),-4.8%,-3.5%,37.9%
